# OIROV Software Track — Session 04
## Sensors & Actuators: IR/Flame, Servo, Relay & Pump

**Trainer:** Hazem Elshafey — CTO, OIROV Robotics  
**Duration:** 2 hours | **Schedule:** 3 weeks, 2 sessions/week  
**Platforms:** Arduino Uno and ESP32  
**Final project:** Firefighting Robot

**Goal:** read sensor signals and structure actuator commands for the agreed IR/flame sensor, servo, relay, and water-pump topics, with clear separation between sensing, decision, and actuation.

**Agenda:** 10 min recap, 25 min sensors, 20 min actuator interfaces, 45 min lab/simulation, 15 min review, 5 min exit ticket.

> Hardware alignment: these lessons follow the agreed Hardware Track sequence. Verify exact board pins, module logic, voltage/current ratings, and wiring against the team's approved hardware setup. Examples are instructional, not a substitute for the wiring plan.

## Learning outcomes
By the end, learners should explain the core concepts, modify the provided C/C++ sketch, test one subsystem at a time, and record expected versus actual behavior.

## 1. Link to Hardware Session 04 — Sensors & Actuators
The agreed Hardware Track includes IR/flame sensors, DC/servo motors, relay, and water pump. Software should read and validate a sensor first, make a decision second, and request an actuator action only when permitted by the system state. Exact module logic, servo interface, relay polarity, and pump ratings must be confirmed from the actual components.

## 2. Read a digital sensor and log it
```cpp
const int flameOutPin = 2; // Example only; verify approved pinout

void setup() {
  Serial.begin(9600);
  pinMode(flameOutPin, INPUT);
}

void loop() {
  int rawState = digitalRead(flameOutPin);
  Serial.print("raw flame output = ");
  Serial.println(rawState);
  delay(100);
}
```
Some modules are active-low; do not decide what `0` or `1` means until you observe the actual module in both conditions.

## 3. Separate raw reading from decision logic
```cpp
bool targetDetectedFromDigitalState(int rawState, bool activeLow) {
  return activeLow ? (rawState == LOW) : (rawState == HIGH);
}
```
This function only illustrates polarity handling. Confirm the sensor output behavior, threshold/potentiometer settings, and environment before relying on it. Avoid treating one noisy sample as sufficient evidence for a safety-related action.

## 4. Servo control — concept and example
A hobby servo typically receives a control signal and its own suitable power supply. The control signal and power requirements depend on the exact servo.

```cpp
#include <Servo.h>
Servo nozzleServo;

void setup() {
  nozzleServo.attach(9); // Example pin; confirm board and wiring
  nozzleServo.write(90); // Example center position
}

void loop() {
  // Move only within the safe mechanical range for the installed servo.
}
```
Do not assume the servo can be powered safely from the board's 5 V pin; check its stall current and the approved power plan.

## 5. Relay and water pump — software interface
A relay module may be active-high or active-low. A pump is an inductive load and must be driven by a correctly rated switching/driver circuit and suitable power supply, not a GPIO pin.

```cpp
const int pumpRelayPin = 4; // Example only
const bool relayActiveLow = true; // Confirm actual module

void setPump(bool on) {
  digitalWrite(pumpRelayPin, (on ^ relayActiveLow) ? HIGH : LOW);
}

void setup() {
  pinMode(pumpRelayPin, OUTPUT);
  setPump(false); // Confirm this maps to OFF on the actual relay
}

void loop() {
  // Pump activation must be governed by tested state logic and time limits.
}
```
Before connecting a pump, verify relay contact rating, supply, wiring, and inductive-load protection with the hardware trainer.

## 6. Interlock the actuator with system state
Do not switch on the pump directly from a raw sensor read. A safer software structure is:
1. Read sensor.
2. Validate input and confirm detection condition.
3. Check current state and any timeout/stop conditions.
4. Stop movement if required by the design.
5. Request pump activation through the approved driver.
6. Limit activation duration and return to a safe state.
7. On invalid input or fault, set outputs to their defined safe state.

## 7. Practical lab — Sensor-to-actuator simulation (45 min)
1. Read a sensor module and print raw readings.
2. Identify active-high/active-low behavior.
3. Implement a detection function.
4. Simulate the pump request using an LED or Serial message first; do not connect the real pump yet.
5. Add a maximum activation-time rule in the state logic.
6. Test target absent, target present, unstable signal, and invalid/disconnected sensor cases.
7. Only after code and wiring review should the trainer supervise an actuator test.

**Deliverable:** sensor polarity note, state/decision table, and simulation test results.

## 8. Review and homework
1. Why should raw sensor readings be separated from decisions?
2. Why must relay polarity be tested?
3. Why is a pump not a GPIO load?
4. What should happen when sensor input becomes invalid during activation?

**Homework:** draw the signal path from sensor output to microcontroller decision to relay/driver input to pump power circuit.

## Course connection
**Previous:** Session 03 — DC Motors, H-Bridge, PWM.  
**Hardware counterpart:** Session 04 — Sensors & Actuators.  
**Next:** Session 05 — Power Distribution, Common Ground, Protection & Wiring.